# 02 - Main experiment matrix - batch 1 (3 seeds per configuration)

**Protocol (fixed for every experiment below):**
- Data: 45k train / 5k val split (`build_loaders_val`); all decisions are made on **val**.
  The final headline number comes later from re-training the winner once on the full 50k.
- Seeds: **0, 42, 123** - every configuration runs all three. Nothing is rerolled or discarded.
  Caveat: seed 42 is known to collapse for *no-BN* configs on this data order (exp_002). If it
  happens again, keep the run and flag it - it is a finding, not an accident to hide.
- Health gate: after each run, val acc < 5% prints a collapse warning; watch the per-epoch prints.
- One cell = one configuration. Legacy runs (exp_000-002, 50k protocol) are excluded from the table.

**How to run:** run the two setup cells first, then any experiment cell independently.


## Setup - imports and GPU check

In [6]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.config import PROJECT_ROOT, DATA_DIR, RUNS_DIR, SEED, CIFAR100_MEAN, CIFAR100_STD
from src.models import baseline_cnn, baseline_cnn_bn, cnn_3x3, cnn_down
from src.data import build_datasets, build_loaders, build_loaders_val
from src.engine import train_one_epoch, evaluate, fit
from src.utils import set_seed, save_run, check_gpu, warmup_cosine
from src.experiments import run_experiment          # the multi-seed harness (was a notebook cell)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

check_gpu()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
cuda NVIDIA GeForce RTX 4060 Laptop GPU
avg ms/matmul: 39  -> FAST, good to go


39.279043674468994

## Shared data loaders (45k / 5k val)

In [7]:
# Shared loaders for all no-augmentation experiments (rebuilding them per cell wastes ~30 s)
train_loader, val_loader, test_loader = build_loaders_val(batch_size=128)
print(f"train {len(train_loader)} batches | val {len(val_loader)} batches")


train 351 batches | val 40 batches


## exp_101 - Baseline architecture, SGD 1e-2 (3 seeds)

In [5]:
# exp_101: the task's baseline architecture, SGD 1e-2 (the original recipe, now on 3 seeds)
run_experiment("exp_101", "baseline_sgd", baseline_cnn,
               lambda p: torch.optim.SGD(p, lr=1e-2),
               extra={"arch": "baseline", "optimizer": "SGD", "lr": 1e-2, "aug": "none"},
               loaders=(train_loader, val_loader))


epoch  1/10 | train 4.597/0.017 | test 4.574/0.018 | 119s
epoch  2/10 | train 4.466/0.027 | test 4.313/0.036 | 119s
epoch  3/10 | train 4.272/0.043 | test 4.242/0.056 | 120s
epoch  4/10 | train 4.224/0.049 | test 4.193/0.055 | 121s
epoch  5/10 | train 4.165/0.059 | test 4.120/0.073 | 122s
epoch  6/10 | train 4.090/0.073 | test 4.047/0.078 | 122s
epoch  7/10 | train 4.017/0.082 | test 3.981/0.090 | 122s
epoch  8/10 | train 3.956/0.090 | test 3.908/0.101 | 122s
epoch  9/10 | train 3.901/0.097 | test 3.877/0.099 | 122s
epoch 10/10 | train 3.855/0.104 | test 3.819/0.117 | 122s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_101_baseline_sgd_s0 | final acc: 0.1166
epoch  1/10 | train 4.599/0.013 | test 4.582/0.018 | 122s
epoch  2/10 | train 4.455/0.031 | test 4.291/0.044 | 122s
epoch  3/10 | train 4.266/0.044 | test 4.237/0.056 | 122s
epoch  4/10 | train 4.217/0.055 | test 4.180/0.065 | 122s
epoch  5/10 | train 4.146/0.066 | test 4.098/0.072 | 122s
epoch  6/10 | train 4.085/0.073 | te

{0: {'train_loss': [4.597405986568527,
   4.465607041307324,
   4.27159753212562,
   4.223674874699693,
   4.164758701949378,
   4.089732691093728,
   4.017155241083215,
   3.9563932303349856,
   3.900840747730005,
   3.8553114277005536],
  'train_acc': [0.016960470085470084,
   0.027021011396011395,
   0.04284633190883191,
   0.0489894943019943,
   0.05931712962962963,
   0.07264957264957266,
   0.08193108974358974,
   0.08976584757834757,
   0.09711093304843305,
   0.1036769943019943],
  'test_loss': [4.574407231807709,
   4.3125142574310305,
   4.242395114898682,
   4.192567408084869,
   4.120017069578171,
   4.0473467946052555,
   3.9808405339717865,
   3.9080975711345673,
   3.877415972948074,
   3.818814992904663],
  'test_acc': [0.0178,
   0.0356,
   0.0558,
   0.0546,
   0.0728,
   0.0778,
   0.09,
   0.1012,
   0.0986,
   0.1166],
  'epoch_time': [118.76816153526306,
   118.63206362724304,
   120.29452300071716,
   121.3171124458313,
   121.57206058502197,
   121.6891517639160

## exp_102 - Adam 1e-3

In [6]:
# exp_102: optimizer -> Adam 1e-3 (single change vs exp_101)
run_experiment("exp_102", "adam", baseline_cnn,
               lambda p: torch.optim.Adam(p, lr=1e-3),
               extra={"arch": "baseline", "optimizer": "Adam", "lr": 1e-3, "aug": "none"},
               loaders=(train_loader, val_loader))


epoch  1/10 | train 4.393/0.026 | test 4.175/0.052 | 119s
epoch  2/10 | train 3.968/0.087 | test 3.781/0.120 | 119s
epoch  3/10 | train 3.638/0.141 | test 3.535/0.157 | 119s
epoch  4/10 | train 3.429/0.176 | test 3.336/0.195 | 119s
epoch  5/10 | train 3.280/0.203 | test 3.213/0.212 | 119s
epoch  6/10 | train 3.161/0.231 | test 3.129/0.235 | 119s
epoch  7/10 | train 3.060/0.251 | test 3.028/0.250 | 119s
epoch  8/10 | train 2.968/0.270 | test 2.913/0.273 | 119s
epoch  9/10 | train 2.879/0.286 | test 2.869/0.279 | 119s
epoch 10/10 | train 2.811/0.300 | test 2.854/0.279 | 119s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_102_adam_s0 | final acc: 0.2794
epoch  1/10 | train 4.633/0.010 | test 4.606/0.010 | 119s
epoch  2/10 | train 4.606/0.010 | test 4.606/0.010 | 118s
epoch  3/10 | train 4.606/0.009 | test 4.606/0.010 | 118s
epoch  4/10 | train 4.606/0.010 | test 4.607/0.009 | 118s
epoch  5/10 | train 4.606/0.009 | test 4.607/0.008 | 119s
epoch  6/10 | train 4.606/0.009 | test 4.607

{0: {'train_loss': [4.392734198828369,
   3.967649366780903,
   3.638490195967193,
   3.428866478792283,
   3.279826632252446,
   3.1614446157743448,
   3.060049514145593,
   2.9677153265374336,
   2.8794860846636303,
   2.8113709983662662],
  'train_acc': [0.025507478632478632,
   0.08680555555555555,
   0.1414485398860399,
   0.1764823717948718,
   0.20279113247863248,
   0.2311030982905983,
   0.2505787037037037,
   0.270477207977208,
   0.2856793091168091,
   0.29959045584045585],
  'test_loss': [4.1753158330917355,
   3.781063050031662,
   3.535267913341522,
   3.3362962782382963,
   3.212545257806778,
   3.128865247964859,
   3.0284944236278535,
   2.9133803248405457,
   2.869201362133026,
   2.8540341317653657],
  'test_acc': [0.0516,
   0.12,
   0.1574,
   0.195,
   0.2116,
   0.2354,
   0.2496,
   0.2734,
   0.2788,
   0.2794],
  'epoch_time': [118.83704161643982,
   118.77090191841125,
   119.16457319259644,
   119.1286895275116,
   119.1027250289917,
   119.0044298171997,
  

## exp_103 - Add BatchNorm

In [7]:
# exp_103: + BatchNorm after every Conv (single change vs exp_102)
run_experiment("exp_103", "bn", baseline_cnn_bn,
               lambda p: torch.optim.Adam(p, lr=1e-3),
               extra={"arch": "baseline+BN", "optimizer": "Adam", "lr": 1e-3, "aug": "none"},
               loaders=(train_loader, val_loader))


epoch  1/10 | train 3.807/0.116 | test 3.426/0.173 | 130s
epoch  2/10 | train 3.174/0.223 | test 3.103/0.228 | 130s
epoch  3/10 | train 2.811/0.290 | test 3.004/0.254 | 130s
epoch  4/10 | train 2.532/0.345 | test 2.540/0.336 | 130s
epoch  5/10 | train 2.313/0.392 | test 2.361/0.384 | 130s
epoch  6/10 | train 2.135/0.429 | test 2.326/0.385 | 129s
epoch  7/10 | train 1.988/0.464 | test 2.205/0.419 | 129s
epoch  8/10 | train 1.848/0.499 | test 1.998/0.467 | 130s
epoch  9/10 | train 1.734/0.524 | test 2.149/0.431 | 130s
epoch 10/10 | train 1.630/0.551 | test 1.954/0.475 | 130s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_103_bn_s0 | final acc: 0.4746
epoch  1/10 | train 3.835/0.111 | test 3.400/0.175 | 130s
epoch  2/10 | train 3.204/0.218 | test 3.094/0.225 | 130s
epoch  3/10 | train 2.836/0.283 | test 2.804/0.281 | 130s
epoch  4/10 | train 2.560/0.340 | test 2.678/0.319 | 130s
epoch  5/10 | train 2.329/0.389 | test 2.406/0.370 | 131s
epoch  6/10 | train 2.148/0.430 | test 2.154/0

{0: {'train_loss': [3.80650891809382,
   3.1735074329919626,
   2.8108586857461524,
   2.5320690812548343,
   2.3128068250129026,
   2.1353192186763144,
   1.9878608806860074,
   1.8478565810412764,
   1.7338572726969705,
   1.6304547569350978],
  'train_acc': [0.11571848290598291,
   0.22268963675213677,
   0.2897079772079772,
   0.3451077279202279,
   0.3923165954415954,
   0.42939814814814814,
   0.4640090811965812,
   0.4985977564102564,
   0.5244836182336182,
   0.55119301994302],
  'test_loss': [3.4257052063941957,
   3.1034383535385133,
   3.003915846347809,
   2.5402720898389815,
   2.3614698320627214,
   2.3259293675422668,
   2.2050823092460634,
   1.9980086624622344,
   2.14870328605175,
   1.9535004317760467],
  'test_acc': [0.1726,
   0.228,
   0.2538,
   0.3362,
   0.3842,
   0.3848,
   0.4192,
   0.4672,
   0.4314,
   0.4746],
  'epoch_time': [130.05115032196045,
   130.26395535469055,
   130.00271701812744,
   129.99576210975647,
   129.73328566551208,
   129.4525082111

## exp_104 - Add warmup + cosine LR schedule

In [ ]:
# exp_104: + lr schedule (2-epoch warmup, then cosine decay) on top of BN + Adam
# (warmup_cosine lives in src/utils.py and is reused by every later experiment.)

# Sanity: the lr curve should rise for 2 epochs, then decay (fit steps the scheduler per epoch)
_opt = torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=1e-3)
_sch = warmup_cosine(_opt, 10)
_lrs = []
for _ in range(10):
    _sch.step(); _lrs.append(_opt.param_groups[0]["lr"])
print("lr per epoch:", [round(v, 5) for v in _lrs])

run_experiment("exp_104", "lrsched", baseline_cnn_bn,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "baseline+BN", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "none"},
               loaders=(train_loader, val_loader))


## exp_105 - Add crop/flip augmentation

In [9]:
# exp_105: + data augmentation (RandomCrop(32, padding=4) + HorizontalFlip) on top of exp_104's recipe.
# Task-doc caveat: under short training augmentation can look like a LOSS - judge it by
# "train_acc down but val not worse" (stronger regularization; retry with longer training later).
from torchvision import transforms
aug_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
aug_loader, aug_val, _ = build_loaders_val(batch_size=128, train_tf=aug_train_tf)

run_experiment("exp_105", "aug", baseline_cnn_bn,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "baseline+BN", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "crop+flip"},
               loaders=(aug_loader, aug_val))


epoch  1/10 | train 4.060/0.104 | test 3.745/0.145 | 132s
epoch  2/10 | train 3.530/0.170 | test 3.418/0.169 | 132s
epoch  3/10 | train 3.114/0.234 | test 3.110/0.236 | 132s
epoch  4/10 | train 2.772/0.297 | test 2.821/0.285 | 132s
epoch  5/10 | train 2.515/0.352 | test 2.534/0.345 | 132s
epoch  6/10 | train 2.315/0.395 | test 2.598/0.357 | 132s
epoch  7/10 | train 2.129/0.437 | test 2.312/0.399 | 132s
epoch  8/10 | train 1.966/0.478 | test 2.117/0.445 | 132s
epoch  9/10 | train 1.829/0.511 | test 1.920/0.477 | 132s
epoch 10/10 | train 1.742/0.535 | test 1.846/0.497 | 132s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_105_aug_s0 | final acc: 0.4974
epoch  1/10 | train 4.076/0.103 | test 3.789/0.138 | 132s
epoch  2/10 | train 3.530/0.171 | test 3.352/0.184 | 132s
epoch  3/10 | train 3.109/0.233 | test 3.034/0.234 | 132s
epoch  4/10 | train 2.766/0.300 | test 2.790/0.287 | 132s
epoch  5/10 | train 2.511/0.354 | test 2.683/0.324 | 132s
epoch  6/10 | train 2.307/0.399 | test 2.629/

{0: {'train_loss': [4.059793338476762,
   3.529601281864351,
   3.1136526334659327,
   2.771805589355295,
   2.515285298355624,
   2.3145836078882898,
   2.1285706683781074,
   1.9660411095007873,
   1.8290898592723401,
   1.7423491871934331],
  'train_acc': [0.10423344017094018,
   0.17036146723646722,
   0.23426371082621084,
   0.29694177350427353,
   0.3521189458689459,
   0.3948762464387464,
   0.4365874287749288,
   0.47774216524216523,
   0.5112624643874644,
   0.5352564102564102],
  'test_loss': [3.7447914481163025,
   3.418136900663376,
   3.1098322451114653,
   2.820960944890976,
   2.5335412561893462,
   2.597752660512924,
   2.3124279737472535,
   2.116657403111458,
   1.9196071922779083,
   1.845585545897484],
  'test_acc': [0.145,
   0.1694,
   0.2364,
   0.285,
   0.3452,
   0.3568,
   0.3992,
   0.4446,
   0.4768,
   0.4974],
  'epoch_time': [132.4034457206726,
   132.16547894477844,
   132.0335807800293,
   131.93547558784485,
   132.01439547538757,
   132.3154308795929

## exp_106 - Replace 5x5 convs with 2x3x3 stacks (aug base)

In [17]:
# exp_106: architecture - every 5x5 conv -> two 3x3 convs (2x3x3 == 5x5 receptive field,
# 18C^2 vs 25C^2 weights, one extra nonlinearity per stage). Task-doc factor 7a.
#
# NOTE: cnn_3x3 in src/models.py is a doc-guided STARTING design. Evolve it (widths, depth,
# residual connections, your own tweaks) and write the "why this design" reasoning yourself -
# the architecture is a graded, personal part of the assignment.
run_experiment("exp_106", "arch_3x3", cnn_3x3,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "2x3x3 stack", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "crop+flip"},
               loaders=(aug_loader, aug_val))


epoch  1/10 | train 4.079/0.094 | test 3.816/0.129 | 139s
epoch  2/10 | train 3.598/0.151 | test 3.473/0.158 | 141s
epoch  3/10 | train 3.114/0.228 | test 3.150/0.218 | 141s
epoch  4/10 | train 2.715/0.300 | test 2.686/0.302 | 140s
epoch  5/10 | train 2.439/0.355 | test 2.534/0.335 | 141s
epoch  6/10 | train 2.204/0.409 | test 2.384/0.383 | 140s
epoch  7/10 | train 1.997/0.458 | test 2.144/0.431 | 139s
epoch  8/10 | train 1.817/0.501 | test 1.852/0.492 | 139s
epoch  9/10 | train 1.672/0.541 | test 1.764/0.518 | 139s
epoch 10/10 | train 1.573/0.569 | test 1.667/0.537 | 139s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_106_arch_3x3_s0 | final acc: 0.5374
epoch  1/10 | train 4.073/0.101 | test 3.773/0.137 | 139s
epoch  2/10 | train 3.579/0.154 | test 3.444/0.172 | 139s
epoch  3/10 | train 3.117/0.225 | test 2.984/0.249 | 139s
epoch  4/10 | train 2.722/0.297 | test 2.913/0.275 | 138s
epoch  5/10 | train 2.446/0.355 | test 2.553/0.340 | 138s
epoch  6/10 | train 2.198/0.408 | test 2

{0: {'train_loss': [4.078502280759675,
   3.5977771479180056,
   3.113641504548554,
   2.7150044441223145,
   2.438865731244753,
   2.203550858035726,
   1.9974426605083324,
   1.8174719970110815,
   1.6719076232692796,
   1.5725406615482775],
  'train_acc': [0.0942619301994302,
   0.15057425213675213,
   0.22800925925925927,
   0.29990206552706555,
   0.35472311253561256,
   0.4088986823361823,
   0.458244301994302,
   0.5010683760683761,
   0.541488603988604,
   0.5689102564102564],
  'test_loss': [3.816131114959717,
   3.47277655005455,
   3.1497165679931642,
   2.6855492025613783,
   2.533987021446228,
   2.3838225305080414,
   2.1438892751932146,
   1.8523201197385788,
   1.763543352484703,
   1.6669879674911499],
  'test_acc': [0.1292,
   0.1582,
   0.2176,
   0.3016,
   0.3346,
   0.3834,
   0.4308,
   0.4924,
   0.5178,
   0.5374],
  'epoch_time': [139.405335187912,
   140.57682132720947,
   140.90263628959656,
   140.40919089317322,
   140.97730374336243,
   139.69377279281616

## exp_107 - Downsampled 3x3 stack (aug base)

In [18]:
# exp_107: architecture - 3x3 stack + stride-2 downsampling (thin stem, stages at 32/16/8/4 px).
# Downsampling cuts compute ~4x per stage, which buys the extra depth at roughly constant cost.
# Same note as exp_106: cnn_down is a starting design - make it yours.
run_experiment("exp_107", "arch_down", cnn_down,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "downsampled", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "crop+flip"},
               loaders=(aug_loader, aug_val))


epoch  1/10 | train 4.003/0.108 | test 3.616/0.153 | 21s
epoch  2/10 | train 3.382/0.184 | test 3.004/0.245 | 21s
epoch  3/10 | train 2.813/0.282 | test 2.866/0.291 | 21s
epoch  4/10 | train 2.349/0.382 | test 2.349/0.392 | 22s
epoch  5/10 | train 2.020/0.448 | test 2.099/0.452 | 23s
epoch  6/10 | train 1.772/0.507 | test 1.807/0.502 | 23s
epoch  7/10 | train 1.560/0.561 | test 1.686/0.528 | 23s
epoch  8/10 | train 1.378/0.608 | test 1.531/0.577 | 24s
epoch  9/10 | train 1.240/0.646 | test 1.424/0.608 | 27s
epoch 10/10 | train 1.150/0.670 | test 1.363/0.619 | 28s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_107_arch_down_s0 | final acc: 0.6194
epoch  1/10 | train 3.996/0.108 | test 3.626/0.166 | 24s
epoch  2/10 | train 3.359/0.189 | test 3.068/0.234 | 28s
epoch  3/10 | train 2.815/0.280 | test 2.611/0.326 | 24s
epoch  4/10 | train 2.347/0.376 | test 2.285/0.394 | 23s
epoch  5/10 | train 2.029/0.448 | test 2.237/0.417 | 28s
epoch  6/10 | train 1.778/0.509 | test 1.829/0.505 | 2

{0: {'train_loss': [4.00277581513777,
   3.3821971891952036,
   2.8131295547865736,
   2.3488298294551013,
   2.020367263728737,
   1.77212317682739,
   1.5601042699270438,
   1.3784214112154098,
   1.239823616807617,
   1.1500163923980844],
  'train_acc': [0.10803952991452992,
   0.18438390313390313,
   0.2819399928774929,
   0.3815215455840456,
   0.44751602564102566,
   0.5073228276353277,
   0.5613425925925926,
   0.6083733974358975,
   0.6460336538461539,
   0.6702056623931624],
  'test_loss': [3.616251748800278,
   3.0037416219711304,
   2.8659638941287993,
   2.349355673789978,
   2.0993528097867964,
   1.8067987978458404,
   1.6863071471452713,
   1.5306849986314774,
   1.4240547388792038,
   1.363148632645607],
  'test_acc': [0.1526,
   0.2448,
   0.2914,
   0.3916,
   0.4522,
   0.502,
   0.5284,
   0.5768,
   0.6076,
   0.6194],
  'epoch_time': [20.712286472320557,
   20.803501844406128,
   20.95812678337097,
   21.655689001083374,
   23.18096947669983,
   23.229267120361328

## exp_108 - 3x3 stack on the no-aug base

In [8]:
# exp_108: architecture - 2x3x3 stack on the **no-aug** base (exp_104's recipe).
# Base correction: exp_105 showed aug is a -4.2 net loss at 10 epochs, so the greedy chain's
# current-best recipe is exp_104, not exp_105. exp_106 was built on the aug base by mistake;
# this run (with exp_104) gives the clean arch x aug 2x2 against exp_106.
# Cost: ~70 min (3 seeds x 10 epochs x ~138 s/epoch).
run_experiment("exp_108", "arch_3x3_noaug", cnn_3x3,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "2x3x3 stack", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "none"},
               loaders=(train_loader, val_loader))


epoch  1/10 | train 4.051/0.102 | test 3.828/0.121 | 137s
epoch  2/10 | train 3.520/0.164 | test 3.253/0.200 | 137s
epoch  3/10 | train 3.010/0.242 | test 3.324/0.201 | 137s
epoch  4/10 | train 2.581/0.326 | test 2.679/0.294 | 136s
epoch  5/10 | train 2.270/0.395 | test 2.363/0.381 | 136s
epoch  6/10 | train 2.003/0.456 | test 2.258/0.413 | 135s
epoch  7/10 | train 1.778/0.511 | test 1.980/0.453 | 135s
epoch  8/10 | train 1.581/0.561 | test 1.755/0.521 | 135s
epoch  9/10 | train 1.419/0.607 | test 1.595/0.567 | 135s
epoch 10/10 | train 1.310/0.640 | test 1.518/0.577 | 135s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_108_arch_3x3_noaug_s0 | final acc: 0.5766
epoch  1/10 | train 4.047/0.105 | test 3.812/0.135 | 135s
epoch  2/10 | train 3.503/0.167 | test 3.252/0.200 | 135s
epoch  3/10 | train 3.018/0.241 | test 3.322/0.193 | 135s
epoch  4/10 | train 2.594/0.326 | test 2.728/0.302 | 135s
epoch  5/10 | train 2.279/0.392 | test 2.380/0.379 | 135s
epoch  6/10 | train 2.020/0.452 | 

{0: {'train_loss': [4.050609608321448,
   3.519872174983011,
   3.0100497927760808,
   2.581243562562513,
   2.270135715476468,
   2.0027577554398452,
   1.7780560679585167,
   1.5812453602114294,
   1.4186668151464217,
   1.3103359204072218],
  'train_acc': [0.10167378917378918,
   0.16370637464387464,
   0.24185363247863248,
   0.32567663817663817,
   0.395477207977208,
   0.45628561253561256,
   0.5106837606837606,
   0.5612535612535613,
   0.6065482549857549,
   0.6402911324786325],
  'test_loss': [3.8278856933116914,
   3.2530755281448362,
   3.324187469482422,
   2.679303008317947,
   2.3632422506809236,
   2.257635790109634,
   1.9798892438411713,
   1.7548909932374954,
   1.5949540853500366,
   1.5181300342082977],
  'test_acc': [0.1206,
   0.2002,
   0.201,
   0.294,
   0.3808,
   0.4132,
   0.4526,
   0.5214,
   0.5672,
   0.5766],
  'epoch_time': [136.72916960716248,
   136.93355011940002,
   137.36639022827148,
   135.54157304763794,
   135.60111618041992,
   135.3071818351

## exp_109 - Downsampled net on the no-aug base

In [9]:
# exp_109: architecture - downsampled net on the **no-aug** base (same base correction as exp_108;
# exp_107 ran it on the aug base). This is the run that decides whether cnn_down is really ahead
# of the plain 3x3 stack or was just less hurt by augmentation.
# Cost: ~13 min (3 seeds x 10 epochs x ~25 s/epoch).
run_experiment("exp_109", "arch_down_noaug", cnn_down,
               lambda p: torch.optim.Adam(p, lr=1e-3), scheduler_fn=warmup_cosine,
               extra={"arch": "downsampled", "optimizer": "Adam", "lr": "1e-3 + warmup/cos", "aug": "none"},
               loaders=(train_loader, val_loader))


epoch  1/10 | train 3.943/0.124 | test 3.573/0.181 | 18s
epoch  2/10 | train 3.170/0.225 | test 2.905/0.276 | 18s
epoch  3/10 | train 2.500/0.344 | test 2.375/0.376 | 18s
epoch  4/10 | train 1.968/0.461 | test 1.962/0.454 | 19s
epoch  5/10 | train 1.599/0.550 | test 1.971/0.466 | 19s
epoch  6/10 | train 1.265/0.637 | test 1.716/0.525 | 18s
epoch  7/10 | train 0.928/0.731 | test 1.679/0.541 | 18s
epoch  8/10 | train 0.595/0.836 | test 1.543/0.581 | 18s
epoch  9/10 | train 0.349/0.925 | test 1.543/0.589 | 18s
epoch 10/10 | train 0.237/0.964 | test 1.538/0.592 | 19s
saved -> C:\Users\sigridwicks\Desktop\DL HW1\runs\exp_109_arch_down_noaug_s0 | final acc: 0.5922
epoch  1/10 | train 3.931/0.123 | test 3.554/0.184 | 18s
epoch  2/10 | train 3.158/0.228 | test 2.939/0.254 | 19s
epoch  3/10 | train 2.515/0.342 | test 2.626/0.335 | 19s
epoch  4/10 | train 1.994/0.456 | test 2.102/0.443 | 18s
epoch  5/10 | train 1.626/0.543 | test 1.859/0.496 | 19s
epoch  6/10 | train 1.298/0.629 | test 1.725/0.5

{0: {'train_loss': [3.9427128795884614,
   3.1698402040704363,
   2.4995076595208583,
   1.968374149072544,
   1.5994454160375133,
   1.2648887708995418,
   0.927798178291049,
   0.5947398191843277,
   0.34880602886194517,
   0.23669550969050482],
  'train_acc': [0.12382033475783476,
   0.22524928774928774,
   0.34375,
   0.46104878917378916,
   0.5497462606837606,
   0.6365963319088319,
   0.7309695512820513,
   0.8360042735042735,
   0.9245459401709402,
   0.9643652065527065],
  'test_loss': [3.5727414429187774,
   2.905189687013626,
   2.3754379510879517,
   1.9622444212436676,
   1.9712255865335464,
   1.715938425064087,
   1.6792097002267838,
   1.543292772769928,
   1.5428400874137878,
   1.5377515792846679],
  'test_acc': [0.1814,
   0.2758,
   0.3762,
   0.4542,
   0.4662,
   0.5254,
   0.5406,
   0.5814,
   0.5888,
   0.5922],
  'epoch_time': [17.98482370376587,
   18.140536785125732,
   18.127254962921143,
   18.555721759796143,
   18.640318632125854,
   18.49989366531372,
  